# 18 — lmfit per-pixel inversion with bi_jax (dask engine)

Companion to NB15 (`15_lmfit_superpixel_bi`) and NB17 (`17_oe_dask_bi`).
Replaces the superpixel compression of NB15 with full per-pixel inversion
tiled through `dask_engine`.

| | NB15 | NB17 | **NB18** |
|---|---|---|---|
| Layer 1 — solver | `lmfit_engine` (LM-LSQ, no priors) | `oe_engine_optx` (JAX OE, with priors) | **`lmfit_engine`** (LM-LSQ, no priors) |
| Layer 2 — image  | `superpixel_engine` (~1 000 superpixels) | `dask_engine` (all pixels, tiled) | **`dask_engine`** (all pixels, tiled) |
| Speed | fast (few superpixels) | fast (JAX vmap) | **slow — serial Python loop per pixel** |

**Forward model**: `bi_jax` (Bi et al. 2023, HEREON water optical model).  
**Free parameters**: `C_0`–`C_7` (phytoplankton classes), `C_Y` (CDOM), `C_ism` (ISM) — 10 total.  
**Scene**: Helsinki EnMAP L2A.

> **Performance note**: `lmfit` is a serial Python loop — each pixel runs
> sequentially with up to `MAX_NFEV` forward-model calls. For ~230 k water
> pixels this takes **tens of minutes to hours** depending on hardware.
> Use `dask_engine`'s `scheduler` parameter to parallelise tiles across
> cores if needed (requires a dask distributed client or `'threads'`
> scheduler; note the GIL limits CPU-bound thread parallelism).

## Imports & configuration

In [ ]:
import numpy as np
import lmfit
import matplotlib.pyplot as plt
import scipy.ndimage as ndi
import time
import xarray as xr
import rioxarray
from pyproj import CRS
from xcube.core.store import new_data_store
import configparser

from bio_optics.water.reflectance import bi_jax
from bio_optics.inversion import lmfit_engine
from bio_optics.image_processing import dask_engine

In [ ]:
NOISE     = 0.001   # Rrs noise level [sr-1] — same as NB15/NB17
MAX_NFEV  = 400     # max lmfit function evaluations per pixel
TILE_SIZE = 65536   # pixels per dask tile (256×256)
SCHEDULER = 'synchronous'  # 'synchronous' = serial; 'threads' = parallel tiles

## Load scene — Helsinki EnMAP S3

In [ ]:
config = configparser.ConfigParser()
config.read('../../config.ini')
credentials = {k: v.strip() for k, v in config['Credentials'].items()}

store = new_data_store(
    's3', max_depth=5, root='coastal-cubes/sek/',
    storage_options=dict(
        anon=False,
        key=credentials['s3_client_id'],
        secret=credentials['s3_client_secret'],
    )
)

INPUT_PREFIX  = 'helsinki/L2A_land/'
OUTPUT_PREFIX = 'helsinki/temp/'

scene_id = 'ENMAP01-____L2A-DT0000158841_20251019T101424Z_002_V010505_20260206T113145Z'

img         = store.open_data(f'{INPUT_PREFIX}{scene_id}.zarr')
scene_crs   = img.rio.crs or CRS.from_wkt(img.spatial_ref.attrs['crs_wkt'])
wavelengths = img.wavelength.values[:80]

refl  = img['reflectance'].isel(band=slice(0, 80))
rrs   = (refl.where(refl > -32768) / 10_000) / np.pi
cloud = (img['cloud'] == 1) | (img['cirrus'] == 1) | (img['haze'] == 1)
rrs   = rrs.where(~cloud)

_wc = store.open_data(f'{OUTPUT_PREFIX}{scene_id}-worldcover.zarr')
_wf = _wc['water_fraction'].values
_labeled, _ = ndi.label(_wf >= 0.5)
_sizes = np.bincount(_labeled.ravel())
_sizes[0] = 0
ocean_mask = xr.DataArray(
    np.isin(_labeled, np.where(_sizes >= 100)[0]),
    coords=_wc['water_fraction'].coords, dims=_wc['water_fraction'].dims,
)
rrs = rrs.where(ocean_mask)

Rrs_arr = rrs.transpose('y', 'x', 'band').values
n_rows, n_cols, n_obs = Rrs_arr.shape
n_water = int(np.isfinite(Rrs_arr).all(axis=-1).sum())

print(f'Image shape : {Rrs_arr.shape}')
print(f'Water pixels: {n_water}')
print(f'Wavelengths : {wavelengths[0]:.1f} – {wavelengths[-1]:.1f} nm ({n_obs} bands)')

## Precompute spectral tables

In [ ]:
pre = bi_jax.precompute(wavelengths)
print(f'n_phy_classes : {pre["n_classes"]}')

## lmfit parameters

Identical to NB15: 10 free parameters, box-constrained (no OE priors).

In [ ]:
params = lmfit.Parameters()

# --- free parameters --------------------------------------------------------
params.add('C_0',   value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('C_1',   value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('C_2',   value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('C_3',   value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('C_4',   value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('C_5',   value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('C_6',   value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('C_7',   value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('C_Y',   value=0.2,  min=0.0,   max=10.0,  vary=True)
params.add('C_ism', value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('offset',value=0.0,  min=-0.02, max=0.02,  vary=False)

# --- CDOM -------------------------------------------------------------------
params.add('S_cdom',        value=0.014, vary=False)
params.add('lambda_0_cdom', value=440.0, vary=False)
params.add('K',             value=0.0,   vary=False)

# --- minerogenic detritus absorption ----------------------------------------
params.add('A_md',        value=0.04,  vary=False)
params.add('S_md',        value=0.011, vary=False)
params.add('C_md',        value=0.0,   vary=False)
params.add('lambda_0_md', value=440.0, vary=False)

# --- biogenic detritus absorption -------------------------------------------
params.add('A_bd',        value=0.001, vary=False)
params.add('S_bd',        value=0.011, vary=False)
params.add('C_bd',        value=0.0,   vary=False)
params.add('lambda_0_bd', value=440.0, vary=False)

# --- detritus attenuation ---------------------------------------------------
params.add('gamma_d',     value=0.5,   vary=False)
params.add('x0',          value=0.96,  vary=False)
params.add('x1',          value=0.5,   vary=False)
params.add('x2',          value=1.0,   vary=False)
params.add('lambda_0_c_d',value=550.0, vary=False)

# --- temperature ------------------------------------------------------------
params.add('T_W',   value=15.0, vary=False)
params.add('T_W_0', value=15.0, vary=False)

# --- phytoplankton packaging ------------------------------------------------
params.add('A_phy',        value=0.06,  vary=False)
params.add('E0',           value=0.65,  vary=False)
params.add('E1',           value=0.67,  vary=False)
params.add('lambda_0_phy', value=440.0, vary=False)

# --- backscattering ratios --------------------------------------------------
for i in range(8):
    params.add(f'b_ratio_C_{i}', value=0.01, vary=False)
params.add('b_ratio_md', value=0.02, vary=False)
params.add('b_ratio_bd', value=0.01, vary=False)

# --- Lee et al. (2011) coefficients ----------------------------------------
params.add('Gw0', value=0.0895, vary=False)
params.add('Gw1', value=0.1247, vary=False)
params.add('Gp0', value=0.0401, vary=False)
params.add('Gp1', value=0.0841, vary=False)

free = [n for n, p in params.items() if p.vary]
print(f'Free parameters ({len(free)}): {free}')

## Forward function adapter

`bi_jax.forward(params, precomputed)` embeds wavelengths inside `precomputed`,
so we wrap it to match the `lmfit_engine` signature `forward_func(params, wavelengths)`.

In [ ]:
def forward_func(params, wavelengths):
    """Adapter: lmfit_engine signature → bi_jax.forward."""
    return np.array(bi_jax.forward(params, pre))

Rrs_test = forward_func(params, wavelengths)
print(f'Forward check — Rrs range: [{Rrs_test.min():.4f}, {Rrs_test.max():.4f}] sr-1')

## Build LmfitSetup

In [ ]:
setup = lmfit_engine.build_inversion(
    params,
    wavelengths,
    forward_func,
    method='least-squares',
    max_nfev=MAX_NFEV,
)
print(f'fit_names : {setup.fit_names}')

## Run — per-pixel lmfit via dask tiling

`dask_engine.invert_image` tiles the image and calls `lmfit_engine.invert_image`
per tile via the shared `invert_fn` interface.  Each pixel within a tile is
inverted sequentially — no JAX, no vmap, no GPU.  This is significantly slower
than NB17 but does not require JAX-differentiable forward models.

Expect **tens of minutes to hours** for the full scene.

In [ ]:
n_tiles = int(np.ceil(n_rows * n_cols / TILE_SIZE))
print(f'Inverting {n_water} water pixels  ({n_rows}×{n_cols} image, '
      f'{len(setup.fit_names)} free params, {n_obs} bands)')
print(f'{n_tiles} tiles of {TILE_SIZE} pixels  —  ~{MAX_NFEV} fwd calls/pixel max')

t0 = time.perf_counter()
results = dask_engine.invert_image(
    Rrs_arr, setup, NOISE,
    invert_fn=lmfit_engine.invert_image,
    tile_size=TILE_SIZE,
    scheduler=SCHEDULER,
    store_chi2_spectral=True,
)
t_total = time.perf_counter() - t0

print(f'Done in {t_total:.1f} s  ({1000*t_total/n_water:.2f} ms/pixel, '
      f'{1000*t_total/n_tiles:.0f} ms/tile)')

## Unpack results

In [ ]:
x_hat     = results['x_hat']          # (n_rows, n_cols, n_fit)
chi2      = results['chi2']           # (n_rows, n_cols)  raw sum of squared residuals
chi2_sp   = results['chi2_spectral']  # (n_rows, n_cols)  plain MSE, no noise weighting
success   = results['success']        # (n_rows, n_cols)  bool
n_nfev    = results['n_nfev']         # (n_rows, n_cols)  int
fit_names = results['fit_names']

water_mask = np.isfinite(Rrs_arr).all(axis=-1)
n_conv     = int(success[water_mask].sum())
n_w        = int(water_mask.sum())

print(f'fit_names  : {fit_names}')
print(f'x_hat shape: {x_hat.shape}')
print(f'converged  : {n_conv} / {n_w} water pixels ({100*n_conv/n_w:.1f}%)')
print(f'median n_nfev (water): {int(np.nanmedian(n_nfev[water_mask]))}')

## Retrieved parameter maps

In [ ]:
_style = {
    'C_Y':   ('YlOrBr', 0, 3,  'C_Y CDOM [1/m]'),
    'C_ism': ('Greys',  0, 50, 'C_ism ISM [g/m³]'),
}

n_params = len(fit_names)
ncols = 5
nrows = int(np.ceil(n_params / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes_flat = axes.ravel() if nrows > 1 else list(axes)

for ax, name in zip(axes_flat, fit_names):
    idx = fit_names.index(name)
    if name in _style:
        cmap, vmin, vmax, title = _style[name]
    else:
        cmap, vmin, vmax, title = 'YlGn', 0, 50, f'{name} [µg/L]'
    im = ax.imshow(x_hat[..., idx], cmap=cmap, vmin=vmin, vmax=vmax, origin='upper')
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

for ax in axes_flat[n_params:]:
    ax.axis('off')

plt.suptitle('bi_jax lmfit per-pixel — retrieved parameters', fontsize=12)
plt.tight_layout()
plt.show()

## Fit diagnostics — chi2, convergence, n_nfev

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

chi2_v = chi2[np.isfinite(chi2)]
im0 = axes[0, 0].imshow(chi2, cmap='plasma', origin='upper',
                         vmin=0, vmax=np.nanpercentile(chi2_v, 98))
plt.colorbar(im0, ax=axes[0, 0], shrink=0.85)
axes[0, 0].set_title(f'chi2 (sum sq. residuals, median={np.nanmedian(chi2_v):.2f})')
axes[0, 0].axis('off')

chi2_sp_v = chi2_sp[np.isfinite(chi2_sp)]
im1 = axes[0, 1].imshow(chi2_sp, cmap='RdYlGn_r', origin='upper',
                         vmin=0, vmax=np.nanpercentile(chi2_sp_v, 98))
plt.colorbar(im1, ax=axes[0, 1], shrink=0.85)
axes[0, 1].set_title(f'chi2_spectral (plain MSE, median={np.nanmedian(chi2_sp_v):.2e})')
axes[0, 1].axis('off')

im2 = axes[0, 2].imshow(success.astype(float), cmap='RdYlGn', origin='upper',
                         vmin=0, vmax=1)
plt.colorbar(im2, ax=axes[0, 2], shrink=0.85)
axes[0, 2].set_title(f'converged ({100*n_conv/n_w:.1f}% of water pixels)')
axes[0, 2].axis('off')

nfev_v = n_nfev[n_nfev >= 0]
im3 = axes[1, 0].imshow(n_nfev.astype(float), cmap='YlOrRd', origin='upper',
                         vmin=0, vmax=MAX_NFEV)
plt.colorbar(im3, ax=axes[1, 0], shrink=0.85)
sat_pct = 100 * (nfev_v >= MAX_NFEV).mean()
axes[1, 0].set_title(f'n_nfev (median={int(np.median(nfev_v))}, '
                     f'at cap={sat_pct:.0f}%)')
axes[1, 0].axis('off')

axes[1, 1].hist(nfev_v, bins=40, color='steelblue', alpha=0.8, density=True)
axes[1, 1].axvline(MAX_NFEV, color='r', ls='--', lw=1.5, label=f'cap={MAX_NFEV}')
axes[1, 1].set_xlabel('n_nfev')
axes[1, 1].set_ylabel('Density')
axes[1, 1].set_title('n_nfev distribution')
axes[1, 1].legend()

axes[1, 2].hist(chi2_sp_v, bins=60, color='darkorange', alpha=0.8, density=True)
axes[1, 2].set_xlabel('chi2_spectral')
axes[1, 2].set_ylabel('Density')
axes[1, 2].set_title('chi2_spectral distribution')

plt.suptitle('lmfit per-pixel diagnostics', fontsize=12)
plt.tight_layout()
plt.show()

## Spot-check — observed vs fitted spectra

Sample 6 pixels: best, worst, and 4 percentile-spaced by `chi2_spectral`.

In [ ]:
chi2_flat = chi2_sp.ravel()
valid_px  = np.where(np.isfinite(chi2_flat))[0]
ranked    = valid_px[np.argsort(chi2_flat[valid_px])]

picks       = [ranked[int(len(ranked) * q)] for q in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]]
pick_labels = ['best', 'p20', 'p40', 'p60', 'p80', 'worst']

Rrs_flat  = Rrs_arr.reshape(-1, n_obs)
x_hat_flat = x_hat.reshape(-1, len(fit_names))

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey=False)
for ax, px_idx, label in zip(axes.ravel(), picks, pick_labels):
    obs  = Rrs_flat[px_idx]
    p_fit = params.copy()
    for name, val in zip(fit_names, x_hat_flat[px_idx]):
        p_fit[name].value = val
    fitted    = forward_func(p_fit, wavelengths)
    chi2_px   = float(chi2_flat[px_idx])

    ax.plot(wavelengths, obs,    'k-',  lw=2, label='observed')
    ax.plot(wavelengths, fitted, 'r--', lw=2, label='fitted')
    ax.fill_between(wavelengths, obs, fitted, alpha=0.15, color='r')
    ax.set_title(f'{label} — chi2_sp={chi2_px:.2e}')
    ax.set_xlabel('λ [nm]')
    ax.set_ylabel('Rrs [sr⁻¹]')
    if label == 'best':
        ax.legend(fontsize=8)

plt.suptitle('Observed vs fitted spectra', fontsize=12)
plt.tight_layout()
plt.show()

## Save to S3 store (optional)

In [ ]:
# coords_2d = {'y': rrs.y, 'x': rrs.x}
# ds = xr.Dataset({
#     'x_hat': xr.DataArray(
#         x_hat, dims=('y', 'x', 'param'),
#         coords={**coords_2d, 'param': fit_names},
#     ),
#     'chi2':           xr.DataArray(chi2,    dims=('y', 'x'), coords=coords_2d),
#     'chi2_spectral':  xr.DataArray(chi2_sp, dims=('y', 'x'), coords=coords_2d),
#     'success':        xr.DataArray(success, dims=('y', 'x'), coords=coords_2d),
#     'n_nfev':         xr.DataArray(n_nfev,  dims=('y', 'x'), coords=coords_2d),
# }).rio.write_crs(scene_crs)
# store.write_data(ds, f'{OUTPUT_PREFIX}{scene_id}-wq_params_lmfit_dask_bi.zarr', replace=True)
# print('Saved.')